In [0]:
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType
from scipy.stats import ks_2samp
from datetime import datetime



# Novos parâmetros para tornar o código genérico
p_inicio = dbutils.widgets.get("data_inicio")
p_fim = dbutils.widgets.get("data_fim")
p_feature_str = dbutils.widgets.get("features")
p_target = dbutils.widgets.get("target")

# p_inicio = "20260101"
# p_fim = "20260110"

# p_feature_str = "renda_mensal_k,tempo_medio_clique_segundos"
# p_target = "comprou_eletronico"

# Processando a string de features para uma lista Python
lista_features = [f.strip() for f in p_feature_str.split(",") if f.strip()]

print(f"Iniciando Job de Monitoramento. Avaliando partições entre {p_inicio} e {p_fim}")
print(f"Monitorando Features: {lista_features}")
print(f"Monitorando Target: {p_target}\n")

nome_tabela_treino = "workspace.default.base_treinamento_modelo"
nome_tabela_prod = "workspace.default.base_tabela_prod"
nome_tabela_auditoria = "workspace.default.log_monitoramento_drift"

# leitura dos dados de treino
df_treino = spark.table(nome_tabela_treino)

# Pushdown filter aproveitando a partição dia_prtc
df_prod = spark.table(nome_tabela_prod).filter(
    F.col("dia_prtc").between(p_inicio, p_fim)
)

qtd_prod = df_prod.count()
if qtd_prod == 0:
    dbutils.notebook.exit(f"Nenhum dado produtivo na janela de {p_inicio} a {p_fim}.")

logs_execucao = []

def registrar(nome, var, val, thresh):
    is_p = "p-value" in nome.lower()
    drift = bool(val < thresh) if is_p else bool(val > thresh)
    logs_execucao.append((datetime.now(), p_inicio, p_fim, nome, var, float(val), float(thresh), drift))

# Feature Drift Dinâmico (Iterando sobre a lista)
# Adicionado sample(0.1) para evitar estouro de memória no Driver ao fazer .collect() 

for feature in lista_features:
    print(f"Calculando KS Test para: {feature}")
    
    # Aplicamos o sample para reduzir o volume
    # Usamos toPandas() para trazer para a memória
    # Extraímos a coluna e convertemos para NumPy array nativo
    arr_treino = df_treino.select(feature).sample(fraction=0.1, seed=42).toPandas()[feature].to_numpy()
    arr_prod = df_prod.select(feature).sample(fraction=0.1, seed=42).toPandas()[feature].to_numpy()

    # O teste KS exige que o array não esteja vazio
    if len(arr_prod) > 0 and len(arr_treino) > 0:
        stat, p_value = ks_2samp(arr_treino, arr_prod)
        registrar("Feature drift KS: p-value", feature, p_value, 0.05)
        

# Concept Drift Dinâmico (Usando o Target) ---
print(f"Calculando Drift de Conversão para o target: {p_target}")
media_t = df_treino.select(F.avg(p_target)).collect()[0][0] or 0
media_p = df_prod.select(F.avg(p_target)).collect()[0][0] or 0

if media_t > 0:
    var_relativa = abs(media_p - media_t) / media_t
    registrar("Concept Drift (Variação Relativa de Conversão)", p_target, var_relativa, 0.15)

schema_log = StructType([
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("periodo_inicio", StringType(), False),
    StructField("periodo_fim", StringType(), False),
    StructField("nome_metrica", StringType(), False),
    StructField("variavel", StringType(), False),
    StructField("valor_calculado", DoubleType(), False),
    StructField("threshold_alerta", DoubleType(), False),
    StructField("drift_detectado", BooleanType(), False)
])

df_logs = spark.createDataFrame(logs_execucao, schema=schema_log)

df_logs.write.format("delta").mode("append").saveAsTable(nome_tabela_auditoria)

print(f"\nMétricas calculadas e registradas com sucesso em {nome_tabela_auditoria}.")